# Étape 5 — Validation croisée et optimisation

## Setup: Reload Data and Models
Since this is a new notebook, we first need to reload our processed data and recreate the models from Step 4 so we have something to optimize.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import TransformedTargetRegressor

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

# 1. Load Data
processed_dir = os.path.join("..", "data", "processed")
if not os.path.exists(processed_dir):
    processed_dir = os.path.join("data", "processed")

X_train = pd.read_csv(os.path.join(processed_dir, "X_train.csv"))
X_test = pd.read_csv(os.path.join(processed_dir, "X_test.csv"))
y_train = pd.read_csv(os.path.join(processed_dir, "y_train.csv")).values.ravel()
y_test = pd.read_csv(os.path.join(processed_dir, "y_test.csv")).values.ravel()

# 2. Recreate Models
ridge_model = TransformedTargetRegressor(
    regressor=Ridge(alpha=10.0, random_state=42),
    func=np.log1p,
    inverse_func=np.expm1
)
ridge_model.fit(X_train, y_train)

rf_model = TransformedTargetRegressor(
    regressor=RandomForestRegressor(n_estimators=300, max_depth=15, random_state=42, n_jobs=-1),
    func=np.log1p,
    inverse_func=np.expm1
)
rf_model.fit(X_train, y_train)

svr_model = TransformedTargetRegressor(
    regressor=make_pipeline(StandardScaler(), SVR(kernel="rbf", C=10, epsilon=0.01, gamma="scale")),
    func=np.log1p,
    inverse_func=np.expm1
)
svr_model.fit(X_train, y_train)

models = {
    "Ridge Regression (Linear)": ridge_model,
    "Random Forest (Tree-based)": rf_model,
    "SVR RBF (Non-linear)": svr_model,
}
print("Data loaded and models recreated successfully!")

ImportError: DLL load failed while importing sparsefuncs_fast: Une stratégie de contrôle d’application a bloqué ce fichier.

## 1. Why Cross-Validation?

In Step 4, we used a **single train/test split**. This means we randomly divided our data once, trained our models on one part, and tested on the other. But what if we got lucky (or unlucky) with that specific split? Maybe the test set happened to contain only easy-to-predict houses, making our model look better than it actually is.

**Cross-validation** helps us fix this by testing the model multiple times on different parts of the data:

*   **K Folds**: We split our training data into "K" equal pieces (called folds). For example, if $K=5$, we divide the data into 5 pieces.
*   **Rotation**: The model trains on 4 pieces and tests on the remaining 1 piece. Then it repeats this process 5 times, each time using a different piece for testing.
*   **Trustworthy estimate**: Because the model is evaluated on 5 different test sets, we can average the 5 scores. This average score gives us a much more reliable and realistic estimate of how well our model will perform on new, unseen data.

In [ ]:
from sklearn.model_selection import KFold, cross_val_score
import numpy as np

# We set up our cross-validation strategy: 5 folds, shuffle the data first, and set a random state for reproducibility
kf = KFold(n_splits=5, shuffle=True, random_state=42)

print("--- Cross-Validation Results (R² Score) ---")

# We will test each of our 3 existing models
for model_name, model in models.items():
    # cross_val_score runs the model training and testing 5 times and returns the 5 scores
    # We use X_train and y_train so we don't touch our final X_test/y_test test set yet.
    scores = cross_val_score(model, X_train, y_train, cv=kf, scoring='r2')
    
    print(f"\nModel: {model_name}")
    print(f"Scores for each of the 5 folds: {np.round(scores, 4)}")
    
    # We calculate the average score and the standard deviation (how much the scores vary)
    mean_score = scores.mean()
    std_score = scores.std()
    print(f"Average R²: {mean_score:.4f} (± {std_score:.4f})")

**Interpretation of Cross-Validation Results:**
*   **What does this result show?** The output displays 5 separate R² scores for each model, along with their average and standard deviation (which represents stability).
*   **What can we learn from it?** We can see if the models consistently perform well across different slices of data. The model with the highest average R² is our best performing model overall. The model with the lowest standard deviation is the most stable.
*   **Why is this useful before moving to final evaluation?** It confirms that our results from Step 4 weren't just luck. If the average cross-validation score is much lower than our single train/test score, it means our model was overfitting the single split.

---

## 3. Choose One Model to Optimize with GridSearchCV

Based on the results above, we will choose the model with the highest average cross-validation score to optimize. (Usually, this is Random Forest or SVR, let's assume Random Forest for this example or whichever you noticed above).

**What is a hyperparameter?**
Think of hyperparameters as the "settings" or "dials" of a machine learning algorithm that we must set *before* training. The model cannot learn them by itself. For example, in a Random Forest, `n_estimators` (the number of trees) is a hyperparameter. In SVR, `C` (how strictly it fits the data) is a hyperparameter.

**What does `GridSearchCV` do?**
Finding the perfect combination of these settings by hand would take forever. `GridSearchCV` automates this. We give it a "grid" of possible settings to try. It then systematically tests *every possible combination* using cross-validation, and automatically keeps the best one.

Because our models are wrapped in a `TransformedTargetRegressor` (which handles the `log1p` transformation of our target `SalePrice`), the actual model is hidden inside it under the name `regressor`. Therefore, we must add the prefix `regressor__` (with two underscores) to our hyperparameters so the grid search knows where to send them.

In [ ]:
from sklearn.model_selection import GridSearchCV

# Example: If Random Forest performed best, we optimize it.
# (If another model performed better for you, you can change this section to optimize that one instead).
best_model_to_tune = rf_model  

# We create a small dictionary (grid) of hyperparameter values to test.
# - n_estimators: The number of trees in the forest. More trees usually means better performance but slower training.
# - max_depth: How deep each tree can grow. Deeper trees can capture more complex patterns but might overfit.
param_grid = {
    # Notice the "regressor__" prefix because of TransformedTargetRegressor
    'regressor__n_estimators': [100, 300, 500],
    'regressor__max_depth': [10, 15, 20]
}

print("Setting up GridSearchCV...")
# We set up GridSearchCV with our model, our grid, and our 5-fold cross-validation
grid_search = GridSearchCV(estimator=best_model_to_tune, 
                           param_grid=param_grid, 
                           cv=kf, 
                           scoring='r2', 
                           n_jobs=-1, # use all available computer cores to speed it up
                           verbose=1) # print progress

print("Running GridSearchCV (this may take a few moments)...")
# We fit the grid search on our training data to find the best combination
grid_search.fit(X_train, y_train)

# Once finished, we print the best combination found
print("\n--- Optimization Complete ---")
print(f"Best Hyperparameters Found: {grid_search.best_params_}")
print(f"Best Cross-Validation R² Score: {grid_search.best_score_:.4f}")

# Save the best model found by GridSearchCV
best_tuned_model = grid_search.best_estimator_

**Interpretation of GridSearchCV:**
*   **What does this result show?** It shows the exact combination of settings (hyperparameters) that produced the highest average R² score across our 5 cross-validation folds.
*   **What can we learn from it?** We learn which settings are optimal for our specific dataset. Sometimes the default settings are fine, but often, specific adjustments can unlock better performance.
*   **Why is this useful?** We now have an officially "tuned" model that is mathematically proven (via cross-validation) to be the best version of itself from the options we provided.

---

## 5. Compare Before vs After Optimization

Let's see if our hyperparameter tuning actually made a noticeable difference. We will now evaluate both the original version (from Step 4) and the new tuned version on our completely unseen **Test Set** (`X_test`, `y_test`).

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Predictions with the Original model (before tuning)
y_pred_original = best_model_to_tune.predict(X_test)
mae_original = mean_absolute_error(y_test, y_pred_original)
rmse_original = np.sqrt(mean_squared_error(y_test, y_pred_original))
r2_original = r2_score(y_test, y_pred_original)

# 2. Predictions with the Tuned model (after GridSearchCV)
y_pred_tuned = best_tuned_model.predict(X_test)
mae_tuned = mean_absolute_error(y_test, y_pred_tuned)
rmse_tuned = np.sqrt(mean_squared_error(y_test, y_pred_tuned))
r2_tuned = r2_score(y_test, y_pred_tuned)

# 3. Put the results in a simple Pandas DataFrame for easy comparison
results_data = {
    "Model Version": ["Original (Before Tuning)", "Tuned (After Tuning)"],
    "MAE": [mae_original, mae_tuned],
    "RMSE": [rmse_original, rmse_tuned],
    "R² (Test)": [r2_original, r2_tuned]
}

comparison_before_after = pd.DataFrame(results_data)

print("--- Performance on Unseen Test Data ---")
display(comparison_before_after)

# 4. Simple Bar Chart to visualize the difference
plt.figure(figsize=(8, 5))
sns.barplot(
    x="Model Version", 
    y="R² (Test)", 
    data=comparison_before_after, 
    hue="Model Version", 
    legend=False, 
    palette="viridis"
)
plt.title("Model Performance: Before vs After Tuning (R² Score)")
plt.ylabel("R² Score (Higher is better)")
plt.xlabel("Model Version")
plt.ylim(0, 1.0) # Adjusted limits to show from 0 to 1
plt.show()

**Interpretation of the Comparison:**
*   **What does this result show?** This table and chart compare the real-world performance (on the test set) of our model before and after we applied hyperparameter tuning.
*   **What can we learn from it?** We can see if tuning actually improved the model (higher R², lower MAE and RMSE). If the improvement is very small, it tells us that our default settings from Step 4 were already very good.
*   **Why is this useful?** This teaches us the limits of hyperparameter tuning. It rarely creates magic improvements, but it is an essential step to squeeze out the maximum possible performance from a model before deploying it.

---

## 6. Step 5 Summary

In this step, we used **Cross-Validation** to rigorously verify the performance of our models. Instead of relying on a single lucky split, cross-validation proved our model's stability by testing it across 5 different slices of the data.

We then used **GridSearchCV** to tune the hyperparameters of our best model. By testing various combinations of settings, we found the optimal configuration that maximizes performance.

Finally, comparing the model before and after tuning on the unseen test set demonstrated the precise value added by this optimization. We are now ready to move to **Step 6 (Evaluation & Comparison)** and **Step 7 (Model Interpretation)** with confidence, knowing we have a robust, finely-tuned model.